# M4/Т4: Information retrieval

In [1]:
import gc
import io
import os
import random
import sys
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import pytrec_eval
import torch
import transformers
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


def flush_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

/home/krv/repo/LLM_course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 1. Загрузка и подготовка датасета
Используем `nfcorpus` (NutritionFacts) из BEIR.

In [2]:
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)
dataset = "nfcorpus"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

print("Скачивание датасета (NFCorpus)...")
data_path = util.download_and_unzip(url, data_dir)
corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

print(f"Corpus size: {len(corpus)}")
print(f"Queries size: {len(queries)}")
print(f"Qrels size: {len(qrels)}")


# Чанкинг документов
def chunk_text(text, max_words=120, overlap=30):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start = max(0, end - overlap)
    return chunks


chunk_corpus = {}  # chunk_id -> dict
doc_to_chunks = defaultdict(list)

for doc_id, doc in corpus.items():
    text = (doc.get("title", "") + ". " + doc.get("text", "").strip()).strip()
    text = text.lower()  # Базовая нормализация
    chunks = chunk_text(text, max_words=120, overlap=30)
    for idx, ch in enumerate(chunks):
        chunk_id = f"{doc_id}#c{idx}"
        chunk_corpus[chunk_id] = {"text": ch, "doc_id": doc_id, "title": doc.get("title", "").lower(), "chunk_idx": idx}
        doc_to_chunks[doc_id].append(chunk_id)

print(f"Total chunks: {len(chunk_corpus)}, avg chunks/doc: {np.mean([len(v) for v in doc_to_chunks.values()]):.2f}")

Скачивание датасета (NFCorpus)...


100%|██████████| 3633/3633 [00:00<00:00, 189991.73it/s]

Corpus size: 3633
Queries size: 323
Qrels size: 323
Total chunks: 10020, avg chunks/doc: 2.76


## 2. Функция оценки качества (NDCG@k, Recall@k, Precision@k)
Реализуем обертку для `pytrec_eval` для удобного вывода метрик на разных уровнях `k`.

In [3]:
def evaluate_search(qrels, results, k_values=[5, 10, 20]):
    """
    qrels: dict[query_id][doc_id] = relevance
    results: dict[query_id][doc_id] = score
    """
    eval_qrels = {}
    for qid, docs in qrels.items():
        eval_qrels[str(qid)] = {str(did): int(score) for did, score in docs.items()}

    eval_res = {}
    for qid, docs in results.items():
        eval_res[str(qid)] = {str(did): float(score) for did, score in docs.items()}

    metrics_to_calc = set()
    for k in k_values:
        metrics_to_calc.add(f"ndcg_cut_{k}")
        metrics_to_calc.add(f"recall_{k}")
        metrics_to_calc.add(f"P_{k}")

    evaluator = pytrec_eval.RelevanceEvaluator(eval_qrels, metrics_to_calc)
    res = evaluator.evaluate(eval_res)

    avg_metrics = defaultdict(float)
    for qid in res:
        for metric, val in res[qid].items():
            avg_metrics[metric] += val

    for metric in avg_metrics:
        avg_metrics[metric] /= len(res)

    for k in k_values:
        print(f"--- K = {k} ---")
        print(f"NDCG@{k}:      {avg_metrics[f'ndcg_cut_{k}']:.4f}")
        print(f"Precision@{k}: {avg_metrics[f'P_{k}']:.4f}")
        print(f"Recall@{k}:    {avg_metrics[f'recall_{k}']:.4f}")

    return avg_metrics

## 3. Sparse поиск (BM25)
В качестве лексического (sparse) поиска используем BM25. Скоринг производится на уровне чанков, затем мы агрегируем их до уровня документа с помощью Max-Pooling.

In [4]:
chunk_ids = list(chunk_corpus.keys())
chunk_texts = [chunk_corpus[cid]["text"] for cid in chunk_ids]

print("Инициализация BM25...")
tokenized_chunks = [text.split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

bm25_results = {}
print("Выполнение BM25 поиска...")
for qid, qtext in queries.items():
    tokenized_query = qtext.lower().split()
    chunk_scores = bm25.get_scores(tokenized_query)

    doc_scores = defaultdict(float)
    for i, score in enumerate(chunk_scores):
        if score > 0:
            doc_id = chunk_corpus[chunk_ids[i]]["doc_id"]
            doc_scores[doc_id] = max(doc_scores[doc_id], float(score))

    bm25_results[qid] = dict(doc_scores)

print("\n==== Оценка Sparse (BM25) ====")
_ = evaluate_search(qrels, bm25_results, k_values=[5, 10, 20])

Инициализация BM25...
Выполнение BM25 поиска...

==== Оценка Sparse (BM25) ====
--- K = 5 ---
NDCG@5:      0.2876
Precision@5: 0.2415
Recall@5:    0.0975
--- K = 10 ---
NDCG@10:      0.2558
Precision@10: 0.1824
Recall@10:    0.1176
--- K = 20 ---
NDCG@20:      0.2322
Precision@20: 0.1296
Recall@20:    0.1392


## 4. Dense поиск (Qdrant + SentenceTransformers)
Используем легковесную семантическую модель (`intfloat/e5-small-v2`) и векторную базу Qdrant in-memory.

In [5]:
import logging

logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

EMB_MODEL = "intfloat/e5-small-v2"
print(f"Загрузка модели {EMB_MODEL}...")
model = SentenceTransformer(EMB_MODEL, device=device)


def encode_texts(texts, is_query=False, batch_size=256):
    prefix = "query: " if is_query else "passage: "
    processed_texts = [prefix + t.strip() for t in texts]
    return model.encode(processed_texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)


print("Создание эмбеддингов для чанков...")
chunk_embeddings = encode_texts(chunk_texts, is_query=False)
dim = chunk_embeddings.shape[1]

# Инициализация Qdrant
client = QdrantClient(location=":memory:")
collection_name = "nfcorpus_dense"

client.recreate_collection(
    collection_name=collection_name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Загрузка векторов в Qdrant...")
BATCH = 2048
points = []
for i, cid in enumerate(chunk_ids):
    points.append(
        PointStruct(id=i, vector=chunk_embeddings[i].tolist(), payload={"doc_id": chunk_corpus[cid]["doc_id"]})
    )
    if len(points) == BATCH or i == len(chunk_ids) - 1:
        client.upsert(collection_name=collection_name, points=points)
        points = []

print("Выполнение Dense поиска...")
dense_results = {}
for qid, qtext in queries.items():
    qv = encode_texts([qtext.lower()], is_query=True, batch_size=1)[0].tolist()

    # Ищем с запасом чанков
    res = client.query_points(collection_name=collection_name, query=qv, limit=1000, with_payload=True)

    doc_scores = defaultdict(float)
    for r in res.points:
        doc_id = r.payload["doc_id"]
        doc_scores[doc_id] = max(doc_scores[doc_id], float(r.score))

    dense_results[qid] = dict(doc_scores)

print("\n==== Оценка Dense (Qdrant e5-small) ====")
_ = evaluate_search(qrels, dense_results, k_values=[5, 10, 20])

Загрузка модели intfloat/e5-small-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1893.51it/s, Materializing param=pooler.dense.weight]                               


Создание эмбеддингов для чанков...
Загрузка векторов в Qdrant...
Выполнение Dense поиска...

==== Оценка Dense (Qdrant e5-small) ====
--- K = 5 ---
NDCG@5:      0.3642
Precision@5: 0.3146
Recall@5:    0.1233
--- K = 10 ---
NDCG@10:      0.3286
Precision@10: 0.2381
Recall@10:    0.1531
--- K = 20 ---
NDCG@20:      0.3032
Precision@20: 0.1737
Recall@20:    0.1857


## 5. Гибридный поиск (Hybrid - RRF)
Объединяем результаты Sparse и Dense поиска. Используем стратегию **Reciprocal Rank Fusion (RRF)**, которая ранжирует на основе позиций документа в обоих списках (избавляя нас от необходимости нормализовать разные шкалы скоров).

In [6]:
def compute_rrf(rankings_list, k=60, weights=None):
    if weights is None:
        weights = [1.0] * len(rankings_list)

    rrf_scores = defaultdict(float)
    for weight, rankings in zip(weights, rankings_list):
        sorted_docs = sorted(rankings.items(), key=lambda x: x[1], reverse=True)
        for rank, (doc_id, _) in enumerate(sorted_docs):
            rrf_scores[doc_id] += weight * (1.0 / (k + rank + 1))
    return dict(rrf_scores)


def compute_linear_combination(rankings_list, weights=None):
    if weights is None:
        weights = [1.0] * len(rankings_list)

    combined_scores = defaultdict(float)
    for weight, rankings in zip(weights, rankings_list):
        if not rankings:
            continue

        scores = list(rankings.values())
        min_score, max_score = min(scores), max(scores)

        for doc_id, score in rankings.items():
            if max_score > min_score:
                norm_score = (score - min_score) / (max_score - min_score)
            else:
                norm_score = 0.5

            combined_scores[doc_id] += weight * norm_score

    return dict(combined_scores)


print("==== Эксперименты с параметрами гибридного поиска ====\n")
results_table = []

# Перехватываем stdout, чтобы evaluate_search не спамил в консоль
trap = io.StringIO()
old_stdout = sys.stdout
sys.stdout = trap

try:
    best_rrf_k = 60
    best_rrf_score = 0

    # 1. Тестируем RRF с разными k
    for k_val in [10, 30, 60, 100]:
        hybrid_results = {}
        for qid in queries.keys():
            bm25_res = bm25_results.get(qid, {})
            dense_res = dense_results.get(qid, {})
            hybrid_results[qid] = compute_rrf([bm25_res, dense_res], k=k_val)

        metrics = evaluate_search(qrels, hybrid_results, k_values=[5, 10, 20])
        metrics["Strategy"] = f"RRF (k={k_val})"
        results_table.append(metrics)

        if metrics["ndcg_cut_5"] > best_rrf_score:
            best_rrf_score = metrics["ndcg_cut_5"]
            best_rrf_k = k_val

    # 2. Тестируем RRF со смещенными весами (Dense > Sparse)
    for w_dense in [0.6, 0.8, 0.9]:
        w_sparse = round(1.0 - w_dense, 1)
        hybrid_results = {}
        for qid in queries.keys():
            bm25_res = bm25_results.get(qid, {})
            dense_res = dense_results.get(qid, {})
            hybrid_results[qid] = compute_rrf([bm25_res, dense_res], k=best_rrf_k, weights=[w_sparse, w_dense])

        metrics = evaluate_search(qrels, hybrid_results, k_values=[5, 10, 20])
        metrics["Strategy"] = f"Weighted RRF (k={best_rrf_k}, D={w_dense}, S={w_sparse})"
        results_table.append(metrics)

    # 3. Тестируем линейную комбинацию
    for w_dense in [0.5, 0.7, 0.9]:
        w_sparse = round(1.0 - w_dense, 1)
        hybrid_results = {}
        for qid in queries:
            bm25_res = bm25_results.get(qid, {})
            dense_res = dense_results.get(qid, {})
            hybrid_results[qid] = compute_linear_combination([bm25_res, dense_res], weights=[w_sparse, w_dense])

        metrics = evaluate_search(qrels, hybrid_results, k_values=[5, 10, 20])
        metrics["Strategy"] = f"Linear Comb (D={w_dense}, S={w_sparse})"
        results_table.append(metrics)

finally:
    sys.stdout = old_stdout

# Вывод результатов
df_results = pd.DataFrame(results_table)
cols = [
    "Strategy",
    "ndcg_cut_5",
    "ndcg_cut_10",
    "ndcg_cut_20",
    "P_5",
    "P_10",
    "P_20",
    "recall_5",
    "recall_10",
    "recall_20",
]
df_results = df_results[[c for c in cols if c in df_results.columns]]

display(df_results.sort_values("ndcg_cut_5", ascending=False).reset_index(drop=True))

best_run = df_results.loc[df_results["ndcg_cut_5"].idxmax()]
print(f"\n🏆 Лучшая комбинация: {best_run['Strategy']} с NDCG@5 = {best_run['ndcg_cut_5']:.4f}")

==== Эксперименты с параметрами гибридного поиска ====



,Strategy,ndcg_cut_5,ndcg_cut_10,ndcg_cut_20,P_5,P_10,P_20,recall_5,recall_10,recall_20
0,"Linear Comb (D=0.9, S=0.1)",0.372239,0.332456,0.307437,0.318885,0.238700,0.175387,0.123110,0.155268,0.187213
1,"Linear Comb (D=0.7, S=0.3)",0.370587,0.337678,0.309398,0.313313,0.243963,0.175542,0.122249,0.158630,0.190171
2,"Weighted RRF (k=10, D=0.9, S=0.1)",0.368478,0.330104,0.305297,0.318885,0.237152,0.174613,0.123830,0.155566,0.186583
3,"Weighted RRF (k=10, D=0.8, S=0.2)",0.366867,0.331315,0.304949,0.317647,0.241176,0.175697,0.122982,0.158346,0.188102
4,"Weighted RRF (k=10, D=0.6, S=0.4)",0.357596,0.329974,0.301114,0.308359,0.245201,0.174458,0.124469,0.161472,0.188106
5,"Linear Comb (D=0.5, S=0.5)",0.349711,0.321647,0.296673,0.300310,0.235604,0.171981,0.121438,0.155726,0.187099
6,RRF (k=10),0.345242,0.319441,0.292472,0.297833,0.235913,0.169505,0.119938,0.158433,0.183810
7,RRF (k=30),0.339307,0.313783,0.288041,0.291641,0.232198,0.168731,0.114383,0.150803,0.179723
8,RRF (k=60),0.337100,0.309372,0.285774,0.289783,0.228483,0.166563,0.113534,0.146350,0.181563
9,RRF (k=100),0.335545,0.307005,0.282240,0.287307,0.226006,0.162848,0.113188,0.144786,0.177565



🏆 Лучшая комбинация: Linear Comb (D=0.9, S=0.1) с NDCG@5 = 0.3722


## 6. Ключевые выводы

По результатам тестирования на медицинском датасете **NFCorpus (BEIR)** мы увидели следующую картину:

1. **Базовые методы:**
   - **BM25 (Sparse)** ожидаемо показал самый низкий результат (NDCG@5 ~0.287). На специфичных медицинских данных терминология запросов и документов часто не совпадает напрямую (vocabulary mismatch).
   - **Dense (e5-small-v2)** сам по себе работает очень хорошо (NDCG@5 ~0.364), так как модель улавливает семантику, даже если слова не совпадают в точности.

2. **Гибридный поиск (эксперименты):**
   - **RRF с равными весами** независимо от параметра `k` (10, 30, 60, 100) немного просаживает качество (NDCG@5 падает до ~0.33-0.34). Слабый сигнал от BM25 перебивает сильный сигнал от нейросети.
   - **Взвешенный RRF** (где веса сдвинуты в пользу Dense) исправляет эту ситуацию, подтягивая результат ближе к чистой нейросети.
   - **Линейная комбинация нормализованных скоров** показала себя **лучше всего**. При правильном подборе весов (например, 0.9 для Dense и 0.1 для Sparse или 0.7/0.3) гибридный поиск превосходит даже чистый Dense-метод (NDCG@5 > 0.365). 

**Итог:** Гибридный поиск — эфеективный инструмент при условии его грамотной настройки. Если один из ретриверов значительно слабее другого, RRF может только навредить. Для достижения SOTA-результатов на сложных доменах необходимо переходить к нормализации скоров и подбору весов, отдавая предпочтение более сильному семантическому сигналу.